## **Formalizing the Cross-Model Complementarity [seen in XGB and D-MPNN]**

We have descriptive and qualitative data that shows:


* **XGBoost is most competitive on familiar chemistry**
* **D-MPNN loses less performance as similarity decreases**
* so the models have complementary failure modes across interpolation vs extrapolation regimes

In this modelu we will formalize the above points.

---

Right now, we already have the visual impression from the bin plots.

![Alt Text](FIGURE_1.png)

**Compute the per-bin model gap**

For each:

* dataset
* split
* similarity bin

compute:

* `delta_auroc = auroc_dmppn - auroc_xgboost`
* `delta_auprc = auprc_dmppn - auprc_xgboost`

This is the most important computation.

Interpretation notes:

* if delta is near zero in high-sim bins, the models are similarly good on familiar chemistry
* if delta grows positive as similarity decreases, D-MPNN is degrading more gracefully OOD
* if delta shrinks or flips near high-sim bins, XGBoost is more competitive there.

In [ ]:
# ============================================
# Complementary failure-mode analysis (ChEMBL-first)
# Inputs:
#   cross_model_ad_bins.csv
#
# Outputs:
#   ad_bin_model_deltas.csv
#   degradation_summary.csv
#   degradation_wide_table.csv
#   figure_delta_auroc_by_bin.png
#   figure_delta_auprc_by_bin.png
# ============================================

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# -----------------------------
# 0) Config
# -----------------------------
INPUT_CSV = Path("cross_model_ad_bins.csv")   # assumes notebook is inside reports/cross_model_comparison
OUTDIR = Path(".")                            # save outputs to current folder
TARGET_DATASETS = ["chembl"]                  # ChEMBL-first analysis
MODEL_A = "chemprop_dmnn"                     # D-MPNN
MODEL_B = "xgboost"                          # baseline

SIM_ORDER = ["<0.3", "0.3-0.5", "0.5-0.7", ">0.7"]
SIM_TO_X = {b: i for i, b in enumerate(SIM_ORDER)}
SPLIT_ORDER = ["cluster", "random", "scaffold"]

# -----------------------------
# 1) Load and basic checks
# -----------------------------
df = pd.read_csv(INPUT_CSV)

required_cols = {
    "dataset", "split_type", "sim_bin", "model",
    "auroc_mean", "auprc_mean"
}
missing = required_cols - set(df.columns)
if missing:
    raise ValueError(f"Missing required columns: {missing}")

df = df.copy()
df["dataset"] = df["dataset"].str.lower()
df["split_type"] = df["split_type"].str.lower()
df["model"] = df["model"].str.lower()

df = df[df["dataset"].isin(TARGET_DATASETS)].copy()

if df.empty:
    raise ValueError(f"No rows found for datasets={TARGET_DATASETS}")

# keep only expected bins/order if present
df["sim_bin"] = pd.Categorical(df["sim_bin"], categories=SIM_ORDER, ordered=True)
df["split_type"] = pd.Categorical(df["split_type"], categories=SPLIT_ORDER, ordered=True)

# -----------------------------
# 2) Pivot to compute per-bin deltas
#    D-MPNN minus XGBoost
# -----------------------------
pivot = df.pivot_table(
    index=["dataset", "split_type", "sim_bin"],
    columns="model",
    values=["auroc_mean", "auprc_mean"],
    aggfunc="first"
)

# flatten columns
pivot.columns = [f"{metric}_{model}" for metric, model in pivot.columns]
pivot = pivot.reset_index()

for col in [f"auroc_mean_{MODEL_A}", f"auroc_mean_{MODEL_B}",
            f"auprc_mean_{MODEL_A}", f"auprc_mean_{MODEL_B}"]:
    if col not in pivot.columns:
        raise ValueError(f"Expected column not found after pivot: {col}")

pivot["delta_auroc_dmppn_minus_xgb"] = (
    pivot[f"auroc_mean_{MODEL_A}"] - pivot[f"auroc_mean_{MODEL_B}"]
)
pivot["delta_auprc_dmppn_minus_xgb"] = (
    pivot[f"auprc_mean_{MODEL_A}"] - pivot[f"auprc_mean_{MODEL_B}"]
)

# optional: carry n if useful
n_pivot = df.pivot_table(
    index=["dataset", "split_type", "sim_bin"],
    columns="model",
    values="n_mean",
    aggfunc="first"
).reset_index()
n_pivot.columns = ["dataset", "split_type", "sim_bin"] + [f"n_mean_{c}" for c in n_pivot.columns[3:]]

pivot = pivot.merge(n_pivot, on=["dataset", "split_type", "sim_bin"], how="left")

delta_cols = [
    "dataset", "split_type", "sim_bin",
    f"auroc_mean_{MODEL_A}", f"auroc_mean_{MODEL_B}",
    "delta_auroc_dmppn_minus_xgb",
    f"auprc_mean_{MODEL_A}", f"auprc_mean_{MODEL_B}",
    "delta_auprc_dmppn_minus_xgb",
    f"n_mean_{MODEL_A}", f"n_mean_{MODEL_B}",
]
delta_table = pivot[delta_cols].sort_values(["dataset", "split_type", "sim_bin"]).reset_index(drop=True)

delta_csv = OUTDIR / "ad_bin_model_deltas.csv"
delta_table.to_csv(delta_csv, index=False)

print(f"Saved per-bin delta table -> {delta_csv}")
print(delta_table.to_string(index=False))

# -----------------------------
# 3) Summarize degradation within each model
#    degradation = AUROC(>0.7) - AUROC(<0.3)
#    same for AUPRC
# -----------------------------
high_bin = ">0.7"
low_bin = "<0.3"

deg_rows = []
for dataset in TARGET_DATASETS:
    for split in SPLIT_ORDER:
        sub = df[(df["dataset"] == dataset) & (df["split_type"] == split)]
        if sub.empty:
            continue

        for model in [MODEL_A, MODEL_B]:
            subm = sub[sub["model"] == model].set_index("sim_bin")

            if high_bin not in subm.index or low_bin not in subm.index:
                continue

            auroc_high = subm.loc[high_bin, "auroc_mean"]
            auroc_low  = subm.loc[low_bin, "auroc_mean"]
            auprc_high = subm.loc[high_bin, "auprc_mean"]
            auprc_low  = subm.loc[low_bin, "auprc_mean"]

            deg_rows.append({
                "dataset": dataset,
                "split": split,
                "model": model,
                "auroc_<0.3": auroc_low,
                "auroc_>0.7": auroc_high,
                "auroc_degradation_>0.7_minus_<0.3": auroc_high - auroc_low,
                "auprc_<0.3": auprc_low,
                "auprc_>0.7": auprc_high,
                "auprc_degradation_>0.7_minus_<0.3": auprc_high - auprc_low,
            })

degradation = pd.DataFrame(deg_rows).sort_values(["dataset", "split", "model"]).reset_index(drop=True)

deg_csv = OUTDIR / "degradation_summary.csv"
degradation.to_csv(deg_csv, index=False)

print(f"\nSaved degradation summary -> {deg_csv}")
print(degradation.to_string(index=False))

# -----------------------------
# 4) Wide degradation table with delta degradation
#    delta degradation = degradation_xgb - degradation_dmppn
#    positive => XGB degraded more
# -----------------------------
wide = degradation.pivot_table(
    index=["dataset", "split"],
    columns="model",
    values=[
        "auroc_<0.3",
        "auroc_>0.7",
        "auroc_degradation_>0.7_minus_<0.3",
        "auprc_<0.3",
        "auprc_>0.7",
        "auprc_degradation_>0.7_minus_<0.3",
    ],
    aggfunc="first"
)

wide.columns = [f"{metric}_{model}" for metric, model in wide.columns]
wide = wide.reset_index()

wide["delta_auroc_degradation_xgb_minus_dmppn"] = (
    wide[f"auroc_degradation_>0.7_minus_<0.3_{MODEL_B}"] -
    wide[f"auroc_degradation_>0.7_minus_<0.3_{MODEL_A}"]
)

wide["delta_auprc_degradation_xgb_minus_dmppn"] = (
    wide[f"auprc_degradation_>0.7_minus_<0.3_{MODEL_B}"] -
    wide[f"auprc_degradation_>0.7_minus_<0.3_{MODEL_A}"]
)

wide_csv = OUTDIR / "degradation_wide_table.csv"
wide.to_csv(wide_csv, index=False)

print(f"\nSaved wide degradation table -> {wide_csv}")
wide.to_string(index=False)

# -----------------------------
# 5) Dedicated figure: delta plot
#    x-axis = similarity bin
#    y-axis = D-MPNN - XGBoost
# -----------------------------
DELTA_LINE_COLOR = "#87CEEB"

def make_delta_plot(delta_df, metric_col, ylabel, title, output_path):
    plot_df = delta_df.copy()
    plot_df["x"] = plot_df["sim_bin"].map(SIM_TO_X)

    datasets = list(plot_df["dataset"].dropna().unique())
    splits = [s for s in SPLIT_ORDER if s in plot_df["split_type"].astype(str).unique()]

    nrows = len(datasets)
    ncols = len(splits)

    fig, axes = plt.subplots(
        nrows=nrows,
        ncols=ncols,
        figsize=(4.8 * ncols, 4.2 * nrows),
        sharey=True,
        squeeze=False
    )

    for i, dataset in enumerate(datasets):
        for j, split in enumerate(splits):
            ax = axes[i, j]
            sub = plot_df[(plot_df["dataset"] == dataset) & (plot_df["split_type"].astype(str) == split)].copy()
            sub = sub.sort_values("sim_bin")

            ax.axhline(0, linestyle="--", linewidth=1)
            ax.plot(
                sub["x"],
                sub[metric_col],
                marker="o",
                linewidth=2,
                color=DELTA_LINE_COLOR
            )

            # annotate values
            for _, r in sub.iterrows():
                ax.text(
                    r["x"],
                    r[metric_col] + 0.005,
                    f"{r[metric_col]:.3f}",
                    ha="center",
                    va="bottom",
                    fontsize=9
                )

            ax.set_xticks(range(len(SIM_ORDER)))
            ax.set_xticklabels(SIM_ORDER, rotation=25)
            ax.set_title(f"{dataset.upper()} / {split.capitalize()}")
            if j == 0:
                ax.set_ylabel(ylabel)
            ax.set_xlabel("Similarity bin")

    fig.suptitle(title, fontsize=18, y=1.02)
    plt.tight_layout()
    plt.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.show()

auroc_fig = OUTDIR / "figure_delta_auroc_by_bin.png"
make_delta_plot(
    delta_df=delta_table,
    metric_col="delta_auroc_dmppn_minus_xgb",
    ylabel="Δ AUROC (D-MPNN − XGBoost)",
    title="Per-bin AUROC advantage across similarity bins",
    output_path=auroc_fig
)
print(f"\nSaved figure -> {auroc_fig}")

auprc_fig = OUTDIR / "figure_delta_auprc_by_bin.png"
make_delta_plot(
    delta_df=delta_table,
    metric_col="delta_auprc_dmppn_minus_xgb",
    ylabel="Δ AUPRC (D-MPNN − XGBoost)",
    title="Per-bin AUPRC advantage across similarity bins",
    output_path=auprc_fig
)
print(f"Saved figure -> {auprc_fig}")

### **Similarity-dependent complementary failure modes of graph and fingerprint models**

We next asked whether the two model classes failed in the same way along the chemical similarity gradient, or whether their relative performance depended on how far the test compound lay from the training manifold. On ChEMBL, the answer was clearly regime-dependent rather than uniform. When test compounds were highly similar to the training set (>0.7 Tanimoto), D-MPNN and XGBoost were close to parity, with only a very small AUROC advantage for D-MPNN across all three split strategies (cluster: +0.012, random: +0.003, scaffold: +0.008). In contrast, in the most novel regime (<0.3 Tanimoto), the D-MPNN advantage was substantially larger (cluster: +0.098, random: +0.084, scaffold: +0.052 AUROC) (Figure X). The same pattern was observed for AUPRC, where the D-MPNN advantage was also largest in the lowest-similarity bin (cluster: +0.110, random: +0.098, scaffold: +0.082) and smallest in the highest-similarity bin (cluster: +0.011, random: +0.010, scaffold: +0.013) (Figure Sx).

This bin-resolved comparison shows that the two models are not simply better or worse in a global sense. Instead, the relative ranking depends on novelty regime. XGBoost remained most competitive on familiar chemistry, where performance was near-matched, but its relative performance deteriorated more rapidly as similarity decreased. D-MPNN, by contrast, retained a larger share of its discrimination under stronger structural shift. This effect was most visible under the cluster and random split settings, where the AUROC gap widened steadily from the most familiar to the most novel bins, and remained directionally consistent under scaffold splitting even though the intermediate bins were less strictly ordered (Figure X).

The same conclusion emerged when degradation was summarized directly as the drop from the highest-similarity bin (>0.7) to the lowest-similarity bin (<0.3). For AUROC, D-MPNN degraded by 0.165, 0.209, and 0.227 points under cluster, random, and scaffold splits, respectively, whereas XGBoost degraded by 0.250, 0.290, and 0.271 points (Table Y). Thus, the AUROC degradation penalty was consistently smaller for D-MPNN, with XGBoost losing an additional 0.085, 0.081, and 0.044 AUROC points across the three split strategies. The same pattern held for AUPRC: D-MPNN degraded by 0.204, 0.175, and 0.285 points, whereas XGBoost degraded by 0.303, 0.262, and 0.354 points, corresponding to excess degradation penalties of 0.099, 0.088, and 0.069 for XGBoost (Table Y).

Taken together, these results indicate complementary failure modes across interpolation and extrapolation regimes. The fingerprint-based XGBoost baseline is strongest when the test compound remains close to known chemistry, but the learned graph representation degrades more gracefully as structural novelty increases. The practical implication is that model choice cannot be separated from novelty regime: the apparent gap between model classes is small in familiar chemical space, but becomes materially larger once predictions are required on compounds that lie further from the training set.

<p align="center">
  <img src="figure_delta_auroc_by_bin.png" alt="AUROC Comparison" width="1200" />
</p>

_Figure 1. Similarity-dependent AUROC advantage of D-MPNN over XGBoost on ChEMBL.
Per-bin AUROC difference (D-MPNN − XGBoost) across cluster, random, and scaffold splits. The horizontal dashed line marks parity between models. In all three split strategies, the D-MPNN advantage is smallest in the most familiar bin (>0.7 Tanimoto similarity) and larger in lower-similarity bins, indicating that the graph model retains a greater relative advantage as compounds move further from the training manifold._

<p align="center">
  <img src="figure_delta_auprc_by_bin.png" alt="AUprc Comparison" width="1200" />
</p>


_Figure 2. Similarity-dependent AUPRC advantage of D-MPNN over XGBoost on ChEMBL.
Per-bin AUPRC difference (D-MPNN − XGBoost) across cluster, random, and scaffold splits. As with AUROC, the D-MPNN advantage is largest in the most novel chemistry bins and smallest in the most familiar bin, showing that the relative ranking advantage of the graph model is amplified under stronger chemical shift._

In [8]:
deg = pd.read_csv("degradation_summary.csv")
deg

,dataset,split,model,auroc_<0.3,auroc_>0.7,auroc_degradation_>0.7_minus_<0.3,auprc_<0.3,auprc_>0.7,auprc_degradation_>0.7_minus_<0.3
0,chembl,cluster,chemprop_dmnn,0.692410,0.856939,0.164529,0.722798,0.926474,0.203676
1,chembl,cluster,xgboost,0.594793,0.844544,0.249752,0.612459,0.915182,0.302722
2,chembl,random,chemprop_dmnn,0.667032,0.876040,0.209009,0.770268,0.945033,0.174765
3,chembl,random,xgboost,0.583358,0.873502,0.290144,0.672748,0.935157,0.262410
4,chembl,scaffold,chemprop_dmnn,0.614850,0.842174,0.227324,0.639466,0.924284,0.284818
5,chembl,scaffold,xgboost,0.563297,0.834621,0.271324,0.557445,0.911681,0.354235


_Table 2. Degradation of discrimination performance from familiar to novel chemistry on ChEMBL.
For each model and split strategy, degradation is defined as performance in the highest-similarity bin (>0.7) minus performance in the lowest-similarity bin (<0.3). Positive values indicate worsening performance with increasing novelty. The final column reports excess degradation for XGBoost relative to D-MPNN, showing that the fingerprint baseline loses more discrimination than the graph model as train–test similarity decreases._

### **Discussion Section (Open)**

The relative performance gap between D-MPNN and XGBoost was similarity-dependent: XGBoost was most competitive on familiar chemistry, whereas D-MPNN showed a larger relative advantage in more structurally novel regions of chemical space.